# Projeto IA - Unidade 1
## Dia 4: Modelagem Supervisionada (Regressão)

Neste notebook, treinaremos 6 algoritmos de regressão no dataset **California Housing** e avaliaremos o desempenho de cada um usando as métricas obrigatórias: MAE, MSE, RMSE e $R^2$.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from time import time

# Pre-processamento
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Algoritmos de Regressão
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor

# Métricas de Regressão
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

### 1. Preparando os Dados (Recap dos Dias 1 e 2)
Carregamento, divisão (80/20) e padronização com StandardScaler.

In [ ]:
# 1. Carregar dados
housing = fetch_california_housing()
X = housing.data
y = housing.target

# 2. Divisão Treino/Teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Padronização (Crucial para SVR e MLP)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Dados Prontos! Treino: {X_train_scaled.shape}, Teste: {X_test_scaled.shape}")

### 2. Instanciando os 6 Modelos Obrigatórios
Definindo os algoritmos com parâmetros adequados para garantir convergência e bom desempenho.

In [ ]:
# Dicionário de modelos para facilitar o loop de treinamento
modelos = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42, n_jobs=-1),
    "XGBoost": XGBRegressor(random_state=42, n_jobs=-1),
    "SVR": SVR(),
    # MLP: Ocultas=(100, 50), Max_iter=500 para garantir convergência
    "MLP (Rede Neural)": MLPRegressor(hidden_layer_sizes=(100, 50), max_iter=500, random_state=42)
}

### 3. Treinamento e Avaliação (A Mágica Acontece)
Aqui iteramos sobre os modelos, medimos o tempo de treino e extraímos MAE, MSE, RMSE e $R^2$.

In [ ]:
resultados = []

print("Iniciando treinamento...")

for nome, modelo in modelos.items():
    print(f"Treinando {nome}...")
    
    # Medir tempo de treino
    start_time = time()
    modelo.fit(X_train_scaled, y_train)
    tempo_treino = time() - start_time
    
    # Fazer predições no conjunto de teste (dados invisíveis)
    y_pred = modelo.predict(X_test_scaled)
    
    # Calcular Métricas
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    
    # Armazenar
    resultados.append({
        "Modelo": nome,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R²": r2,
        "Tempo (s)": round(tempo_treino, 2)
    })

print("Treinamento concluído!")

### 4. Tabela de Resultados Comparativos
Organizando os resultados em um DataFrame do Pandas para facilitar a análise.

In [ ]:
df_resultados = pd.DataFrame(resultados)

# Ordenar pelo melhor R² (maior é melhor)
df_resultados = df_resultados.sort_values(by="R²", ascending=False).reset_index(drop=True)

display(df_resultados)

# Gráfico comparativo do R²
plt.figure(figsize=(10, 5))
sns.barplot(data=df_resultados, x="R²", y="Modelo", palette="viridis")
plt.title("Comparação de Desempenho (R²) - Modelos de Regressão")
plt.xlim(0, 1)
plt.show()